# 06 - Person-to-Person Variability in Trisomy 21

**Modeling Individual Differences using CGCS**  
**Allen Lab Collaboration**

In [ ]:
# ================================================
# SETUP - Run this first
# ================================================
import sys
from pathlib import Path

if "/content/allen-lab-report-tool" not in str(Path.cwd()):
    repo_path = "/content/allen-lab-report-tool"
    if Path(repo_path).exists():
        %cd /content/allen-lab-report-tool
    else:
        print("Cloning repo...")
        !git clone https://github.com/thinkthoughts/allen-lab-report-tool.git
        %cd allen-lab-report-tool

sys.path.insert(0, str(Path.cwd() / "src"))

import grok
from grok.trisomy_metrics import trisomy_cgcs_score, simulate_intervention_recovery
from grok.visualization import plot_cgcs_vs_noise

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Person-to-Person Variability Notebook Ready")
print(f"Version: {grok.__version__}")

## 1. Simulate Population Variability

In [ ]:
np.random.seed(42)
n_individuals = 200

# Individual variation in overexpression and dysregulation
overexpression = np.random.normal(1.5, 0.15, n_individuals)
dysregulation = np.random.normal(0.4, 0.12, n_individuals)

cgcs_scores = []

for i in range(n_individuals):
    score = trisomy_cgcs_score(
        dosage_ratio=overexpression[i],
        overexpression_imbalance=abs(overexpression[i] - 1.0),
        global_dysregulation=dysregulation[i]
    )['cgcs']
    cgcs_scores.append(score)

cgcs_scores = np.array(cgcs_scores)

print(f"Simulated {n_individuals} individuals")
print(f"Mean CGCS: {cgcs_scores.mean():.4f}")
print(f"Range: {cgcs_scores.min():.4f} — {cgcs_scores.max():.4f}")

## 2. Distribution of CGCS Across Individuals

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(cgcs_scores, bins=25, kde=True, color='purple')
plt.axvline(x=cgcs_scores.mean(), color='red', linestyle='--', label='Mean CGCS')
plt.title('Distribution of CGCS Across Simulated Individuals with Trisomy 21')
plt.xlabel('CGCS Score')
plt.ylabel('Number of Individuals')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Intervention Response Variability

In [ ]:
recovery_strength = 0.45
recovered_scores = [simulate_intervention_recovery(s, recovery_strength) for s in cgcs_scores]

plt.figure(figsize=(10, 6))
plt.scatter(cgcs_scores, recovered_scores, alpha=0.7, color='green')
plt.plot([0,1], [0,1], 'r--', label='No Improvement Line')
plt.title('Individual CGCS Before vs After Intervention')
plt.xlabel('Baseline CGCS')
plt.ylabel('Post-Intervention CGCS')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Summary Statistics

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Mean CGCS', 'Median CGCS', 'Low CGCS (<0.4)', 'High CGCS (>0.7)', 'Mean Improvement'],
    'Value': [
        f"{cgcs_scores.mean():.4f}",
        f"{np.median(cgcs_scores):.4f}",
        f"{(cgcs_scores < 0.4).sum()} individuals",
        f"{(cgcs_scores > 0.7).sum()} individuals",
        f"{np.mean(recovered_scores) - cgcs_scores.mean():.4f}"
    ]
})

display(summary)

---
**Insight**:  
Person-to-person variability is naturally captured by CGCS. This framework can help stratify individuals and predict differential response to admissible interventions — highly relevant for personalized approaches in Down syndrome research.